# Folium

## Imports

In [ ]:
import json

import folium
import geopandas

## Réglages initiaux et affichage d'une carte

Récupérez les coordonnées GPS de Nantes (par exemple sur Wikipédia), puis affichez une carte Folium centrée sur ces coordonnées avec un niveau de zoom permettant de bien voir la ville.

In [ ]:
# Votre code ici

### Solution

In [ ]:
NANTES_COORDS = 47.218371, -1.553621

folium.Map(location=NANTES_COORDS, zoom_start=14)

## Ajout de marqueurs

Maintenant que nous disposons d'une carte correctement initialisée, nous allons y afficher des informations d'utilisation du service de vélos en libre service nantais : bicloo.

Commençons par récupérer les données avec les cellule ci-dessous.

In [ ]:
!wget 'https://data.nantesmetropole.fr/api/records/1.0/search/?dataset=244400404_stations-velos-libre-service-nantes-metropole-disponibilites&q=&rows=1000' -O infos-velos.json

### Chargement des données

Ces données sont maintenant disponibles dans le fichier `infos-velos.json`. Chargez ce fichier à l'aide du module `json` de la bibliothèque standard et affichez les données chargées.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
with open("infos-velos.json", encoding="utf8") as fh:
  content = json.load(fh)
print(content)

### Création des marqueurs

Vous pouvez maintenant itérer sur les données (en particulier sur le contenu associé à la clef `records` de chaque élément des données) pour afficher un marqueur à chaque position de station bicloo.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=14)

for record in content["records"]:
  folium.Marker(location=record["fields"]["position"]).add_to(m)

m

### Personnalisation des marqueurs

Ajoutez un peu d'information à ces marqueurs :

- La couleur doit dépendre du nombre de vélos :
    - 0 → rouge
    - < 4 → jaune
    - sinon vert
- Le nom d'une station s'affiche quand on clique sur un marqueur.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=14)

for record in content["records"]:
  if record["fields"]["status"] != "OPEN":
    color = "black"
  else:
    match record["fields"]["available_bikes"]:
      case x if x > 3:
        color = "green"
      case x if x > 0:
        color = "yellow"
      case _:
        color = "red"
  folium.Marker(
    location=record["fields"]["position"],
    popup=record["fields"]["name"],
    icon=folium.Icon(color=color),
  ).add_to(m)

m

## Ajout d'une couche GeoJSON

Nous allons maintenant créer une carte plus complexe, avec une couche GeoJSON.

Nous allons pour cela récupérer deux fichiers GeoJSON : l'un contient tous les bureaux de vote de Nantes, l'autre les zones de vote.

In [ ]:
!wget 'https://data.nantesmetropole.fr/api/explore/v2.1/catalog/datasets/244400404_decoupage-geographique-bureaux-vote-nantes/exports/geojson?lang=fr&timezone=Europe%2FBerlin' -O zones.geojson
!wget 'https://data.nantesmetropole.fr/api/explore/v2.1/catalog/datasets/244400404_lieux-vote-nantes-metropole/exports/geojson?lang=fr&timezone=Europe%2FBerlin' -O bureaux.geojson

### Chargement des données

Commencez par charger les données récupérées à l'aide de la fonction [`geopandas.read_file`](https://geopandas.org/en/stable/docs/reference/api/geopandas.read_file.html).

Effectuez ensuite une jointure entre les deux tables pour une utilisation future. Vous pourrez utiliser la colonne `lien_lieu_vote` de la table des zones et la colonne `idlieu_vote` de la table des bureaux. La fonction de jointure est [`pandas.DataFrame.join`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html).

In [ ]:
# Votre code ici

#### Solution

In [ ]:
bureaux_gdf = geopandas.read_file("bureaux.geojson")
zones_gdf = geopandas.read_file("zones.geojson")
fat_gdf = zones_gdf.join(bureaux_gdf.set_index("idlieu_vote"),
                         on="lien_lieu_vote",
                         rsuffix="_bureau")

### Affichage des zones de vote

Utilisez [`folium.GeoJson`](https://python-visualization.github.io/folium/modules.html#folium.features.GeoJson) pour afficher les zones de vote.

Si vous souhaitez corser l'exercice, ajoutez l'affichage du nom de la zone et du numéro de bureau lié à la zone au survol de la souris.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
    fat_gdf.drop(columns=["geometry_bureau"]).set_geometry("geometry"),
    tooltip=folium.GeoJsonTooltip(
        fields=["nom", "numero_bureau"],
        aliases=["Zone", "N° de bureau"]),
    name="Zones de vote"
).add_to(m)

folium.LayerControl().add_to(m)

m

### Ajout de marqueurs pour les bureaux de vote

Repartez du code de la carte précédente et ajoutez des marqueurs pour chaque bureau de vote. Pour corser l'exercice, ajoutez l'affichage du nom du bureau et de son adresse quand on clique sur le marqueur.

In [ ]:
# Votre code ici

#### Solution

In [ ]:
m = folium.Map(location=NANTES_COORDS, zoom_start=13)

folium.GeoJson(
    fat_gdf.drop(columns=["geometry_bureau"]).set_geometry("geometry"),
    tooltip=folium.GeoJsonTooltip(
        fields=["nom", "numero_bureau"],
        aliases=["Zone", "N° de bureau"]),
    name="Zones de vote"
    ).add_to(m)

folium.LayerControl().add_to(m)

for _, row in bureaux_gdf.iterrows():
  folium.Marker(
    location=(row.geometry.y, row.geometry.x),
    popup=f"{row.nom}<br />{row.adresse}",
    icon=folium.Icon(icon="info-sign", color="green"),
  ).add_to(m)

m